# Milestone 5: Memory coalescing

## The idea, as a picture in words

Think of GPU memory as a **warehouse** where items are only handed out in **sealed boxes of 32 bytes** (8 floats per box). The GPU never fetches a single float; it always fetches whole boxes.

GPU threads don't work alone. They're grouped into teams of **32 threads called a warp**, and all 32 run the same instruction at the same moment. When a warp executes a line like `value = image[index];`, the hardware collects the 32 addresses and fetches every box that contains at least one of them.

- **Coalesced:** the 32 threads want 32 *neighbouring* floats (128 bytes in a row). That's exactly **4 boxes**, and every byte delivered gets used.
- **Uncoalesced:** the 32 threads want floats that are far apart, say one per image row. That's **32 separate boxes**, and only 4 bytes of each 32-byte box are used. The other 28 bytes are hauled across the memory bus for nothing. That's up to **8× more traffic** for the same work.

Rule of thumb: **neighbouring threads (consecutive `threadIdx.x`) should read neighbouring addresses.**

## Checking our kernels

How a warp is formed: CUDA numbers the threads in a block x-first. With **16 × 16 blocks**, one warp of 32 threads is **two half-rows of 16 pixels**, in two different image rows.

| Kernel | Reads | Writes | Verdict |
|---|---|---|---|
| black level | uint16, consecutive columns | float, consecutive | ✅ coalesced |
| demosaic | Bayer floats, consecutive (±1 neighbours) | `rgb[p*3 + c]`, **every 3rd float** | ⚠️ writes spread 3× |
| white balance | `rgb[p*3 + c]`, **every 3rd float** | same | ⚠️ reads and writes spread 3× |
| denoise | 27 reads, **every 3rd float** | every 3rd float | ⚠️ spread 3× |
| gamma | every 3rd float | bytes, every 3rd byte | ⚠️ spread 3× |

**Problem 1: interleaved RGB.** In `R G B R G B ...`, when 32 threads each read *their* red value, they touch floats 0, 3, 6, ..., 93. That's spread over 3× the memory they need, so each request has to fetch about 3× as many boxes. **Fix: planar layout** `R R R ... G G G ... B B B ...`. Now 32 threads reading red touch floats 0, 1, 2, ..., 31: perfectly coalesced.

**Problem 2: 16-wide blocks.** A warp covers two separate 64-byte pieces of two rows instead of one 128-byte piece. Each piece is still made of whole boxes, so this is a smaller effect, but it doubles the number of separate places a warp asks for. **Fix: 32 × 8 blocks** (still 256 threads), so a warp is one full row segment of 32 pixels.

**An honest prediction:** the stride-3 problem may hurt *less* than the textbook says. The GPU's on-chip caches (L1 and L2) keep the boxes fetched for the red channel, so the green and blue reads moments later often find them already there. **The measurement decides**, which is why we run experiments rather than trust predictions.

## The experiment

| Config | Layout | Block | Why it's included |
|---|---|---|---|
| A | interleaved | 1 × 32 | **Deliberately bad.** A warp is 32 pixels going *down* a column, so every thread is in a different row: fully uncoalesced. It shows what the problem really costs. |
| B | interleaved | 16 × 16 | Milestone 4, our starting point |
| C | interleaved | 32 × 8 | only the block-shape fix |
| D | planar | 16 × 16 | only the layout fix |
| E | planar | 32 × 8 | both fixes |

Config A uses blocks of 32 threads instead of 256, which also changes how the GPU schedules work. So A is a demonstration, not a perfectly fair comparison. B to E are directly comparable.

**Runtime must be T4 GPU.**

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project

## Step 2: Write the new files
- `gpu_planar_kernels.h` / `.cu`: stages 2–5 for the planar layout. The maths is identical; only the indexing changes.
- `gpu_coalescing_experiment_main.cu`: runs and times configurations A to E.

In [ ]:
%%writefile gpu_planar_kernels.h
// gpu_planar_kernels.h
// Stages 2 to 5 rewritten for a PLANAR colour layout (Milestone 5).
//
// Interleaved layout (Milestones 1-4):  R G B R G B R G B ...
// Planar layout (this file):            R R R ... then G G G ... then B B B ...
//   red   of pixel p is at rgbPlanes[p]
//   green of pixel p is at rgbPlanes[pixelCount + p]
//   blue  of pixel p is at rgbPlanes[2 * pixelCount + p]
// It is still ONE buffer of the same size; only the order of values changes.
// Stage 1 is unchanged (the Bayer image has one value per pixel either way).

#pragma once

__global__ void demosaicPlanarKernel(const float* normalisedBayer, float* rgbPlanes,
                                     int imageWidth, int imageHeight);

__global__ void whiteBalancePlanarKernel(float* rgbPlanes, int imageWidth, int imageHeight,
                                         float redGain, float greenGain, float blueGain);

__global__ void denoisePlanarKernel(const float* rgbPlanes, float* denoisedPlanes,
                                    int imageWidth, int imageHeight);

// Reads planar floats, writes INTERLEAVED bytes, because the output file
// (and the CPU reference) store R G B together for each pixel.
__global__ void gammaPlanarKernel(const float* rgbPlanes, unsigned char* outputBytes,
                                  int imageWidth, int imageHeight);


In [ ]:
%%writefile gpu_planar_kernels.cu
// gpu_planar_kernels.cu
// Same maths as the naive kernels. The ONLY change is where each colour value
// lives in memory (see gpu_planar_kernels.h). Why it matters:
// in the interleaved layout, 32 neighbouring threads reading "their red value"
// touch addresses 0, 3, 6, 9 ... (every third float), spread over 3x more
// memory than they need. In the planar layout they read addresses 0, 1, 2, 3 ...
// which is one tight, contiguous block: a coalesced access.

#include "gpu_planar_kernels.h"

// Same helpers as in gpu_neighbourhood_kernels.cu. "static" keeps these copies
// private to this file, so their names do not clash with the ones there.
static __device__ int reflectIndexPlanar(int index, int size) {
    if (index < 0) {
        return -index;
    }
    if (index >= size) {
        return 2 * size - 2 - index;
    }
    return index;
}

static __device__ int clampIndexPlanar(int index, int size) {
    if (index < 0) {
        return 0;
    }
    if (index >= size) {
        return size - 1;
    }
    return index;
}

static __device__ float readBayerPlanar(const float* bayerImage, int imageWidth, int imageHeight,
                                        int row, int column) {
    int safeRow = reflectIndexPlanar(row, imageHeight);
    int safeColumn = reflectIndexPlanar(column, imageWidth);
    return bayerImage[safeRow * imageWidth + safeColumn];
}

__global__ void demosaicPlanarKernel(const float* normalisedBayer, float* rgbPlanes,
                                     int imageWidth, int imageHeight) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    float centre = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row, column);
    float up = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row - 1, column);
    float down = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row + 1, column);
    float left = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row, column - 1);
    float right = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row, column + 1);
    float upLeft = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row - 1, column - 1);
    float upRight = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row - 1, column + 1);
    float downLeft = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row + 1, column - 1);
    float downRight = readBayerPlanar(normalisedBayer, imageWidth, imageHeight, row + 1, column + 1);

    float crossAverage = (up + down + left + right) / 4.0f;
    float diagonalAverage = (upLeft + upRight + downLeft + downRight) / 4.0f;
    float horizontalAverage = (left + right) / 2.0f;
    float verticalAverage = (up + down) / 2.0f;

    bool isEvenRow = (row % 2 == 0);
    bool isEvenColumn = (column % 2 == 0);

    float redValue = 0.0f;
    float greenValue = 0.0f;
    float blueValue = 0.0f;

    if (isEvenRow && isEvenColumn) {
        redValue = centre;
        greenValue = crossAverage;
        blueValue = diagonalAverage;
    } else if (isEvenRow && !isEvenColumn) {
        redValue = horizontalAverage;
        greenValue = centre;
        blueValue = verticalAverage;
    } else if (!isEvenRow && isEvenColumn) {
        redValue = verticalAverage;
        greenValue = centre;
        blueValue = horizontalAverage;
    } else {
        redValue = diagonalAverage;
        greenValue = crossAverage;
        blueValue = centre;
    }

    int pixelCount = imageWidth * imageHeight;
    int pixelIndex = row * imageWidth + column;
    rgbPlanes[pixelIndex] = redValue;
    rgbPlanes[pixelCount + pixelIndex] = greenValue;
    rgbPlanes[2 * pixelCount + pixelIndex] = blueValue;
}

__global__ void whiteBalancePlanarKernel(float* rgbPlanes, int imageWidth, int imageHeight,
                                         float redGain, float greenGain, float blueGain) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    int pixelCount = imageWidth * imageHeight;
    int redIndex = row * imageWidth + column;
    int greenIndex = pixelCount + redIndex;
    int blueIndex = 2 * pixelCount + redIndex;

    float redValue = rgbPlanes[redIndex] * redGain;
    float greenValue = rgbPlanes[greenIndex] * greenGain;
    float blueValue = rgbPlanes[blueIndex] * blueGain;

    if (redValue > 1.0f) {
        redValue = 1.0f;
    }
    if (greenValue > 1.0f) {
        greenValue = 1.0f;
    }
    if (blueValue > 1.0f) {
        blueValue = 1.0f;
    }

    rgbPlanes[redIndex] = redValue;
    rgbPlanes[greenIndex] = greenValue;
    rgbPlanes[blueIndex] = blueValue;
}

__global__ void denoisePlanarKernel(const float* rgbPlanes, float* denoisedPlanes,
                                    int imageWidth, int imageHeight) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    const float gaussianWeights[3][3] = {
        {1.0f, 2.0f, 1.0f},
        {2.0f, 4.0f, 2.0f},
        {1.0f, 2.0f, 1.0f}
    };
    const float weightSum = 16.0f;

    int pixelCount = imageWidth * imageHeight;
    int pixelIndex = row * imageWidth + column;

    for (int channel = 0; channel < 3; channel++) {
        // Each channel is its own plane: move to the start of that plane.
        int planeStart = channel * pixelCount;
        float weightedTotal = 0.0f;

        for (int rowOffset = -1; rowOffset <= 1; rowOffset++) {
            for (int columnOffset = -1; columnOffset <= 1; columnOffset++) {
                int neighbourRow = clampIndexPlanar(row + rowOffset, imageHeight);
                int neighbourColumn = clampIndexPlanar(column + columnOffset, imageWidth);
                int neighbourPixel = neighbourRow * imageWidth + neighbourColumn;
                float neighbourValue = rgbPlanes[planeStart + neighbourPixel];
                float weight = gaussianWeights[rowOffset + 1][columnOffset + 1];
                weightedTotal = weightedTotal + neighbourValue * weight;
            }
        }

        denoisedPlanes[planeStart + pixelIndex] = weightedTotal / weightSum;
    }
}

static __device__ unsigned char gammaToBytePlanar(float linearValue) {
    if (linearValue < 0.0f) {
        linearValue = 0.0f;
    }
    if (linearValue > 1.0f) {
        linearValue = 1.0f;
    }
    float gammaValue = powf(linearValue, 1.0f / 2.2f);
    float scaledValue = gammaValue * 255.0f + 0.5f;
    return static_cast<unsigned char>(scaledValue);
}

__global__ void gammaPlanarKernel(const float* rgbPlanes, unsigned char* outputBytes,
                                  int imageWidth, int imageHeight) {
    int column = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    int pixelCount = imageWidth * imageHeight;
    int pixelIndex = row * imageWidth + column;
    for (int channel = 0; channel < 3; channel++) {
        float linearValue = rgbPlanes[channel * pixelCount + pixelIndex];
        outputBytes[pixelIndex * 3 + channel] = gammaToBytePlanar(linearValue);
    }
}


In [ ]:
%%writefile gpu_coalescing_experiment_main.cu
// gpu_coalescing_experiment_main.cu
// Milestone 5: measures how memory access patterns change kernel speed.
// Five configurations, each timed the Milestone 4 way (1 warm-up + 10 averaged):
//   A. interleaved RGB,  1 x 32 blocks  - deliberately BAD (uncoalesced demo)
//   B. interleaved RGB, 16 x 16 blocks  - Milestone 4, our starting point
//   C. interleaved RGB, 32 x  8 blocks  - a warp now covers one full row segment
//   D. planar RGB,      16 x 16 blocks  - colours stored in separate planes
//   E. planar RGB,      32 x  8 blocks  - both fixes together
// Every configuration uses the same 256 threads per block except A (32), so
// the comparison is mostly about WHICH addresses a warp touches together.

#include <iomanip>
#include <iostream>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_buffers.h"
#include "gpu_kernels.h"
#include "gpu_planar_kernels.h"
#include "isp_io.h"

const int STAGE_COUNT = 5;
const int TIMED_RUN_COUNT = 10;
const int CONFIGURATION_COUNT = 5;

// Runs the 5 stages once, using either the interleaved kernels (Milestone 2)
// or the planar kernels (this milestone). Fills stageMilliseconds[0..4].
static void runPipelineOnce(DeviceBuffers& buffers, const ImageMetadata& metadata,
                            bool usePlanarLayout, dim3 gridShape, dim3 blockShape,
                            cudaEvent_t stageEvents[], float stageMilliseconds[]) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;

    checkCudaError(cudaEventRecord(stageEvents[0]), "recording event 0");

    // Stage 1 is identical for both layouts.
    blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");
    checkCudaError(cudaEventRecord(stageEvents[1]), "recording event 1");

    if (usePlanarLayout) {
        demosaicPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                        imageWidth, imageHeight);
    } else {
        demosaicKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                  imageWidth, imageHeight);
    }
    checkCudaError(cudaGetLastError(), "launching demosaic");
    checkCudaError(cudaEventRecord(stageEvents[2]), "recording event 2");

    if (usePlanarLayout) {
        whiteBalancePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                            metadata.whiteBalanceRed,
                                                            metadata.whiteBalanceGreen,
                                                            metadata.whiteBalanceBlue);
    } else {
        whiteBalanceKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                      metadata.whiteBalanceRed,
                                                      metadata.whiteBalanceGreen,
                                                      metadata.whiteBalanceBlue);
    }
    checkCudaError(cudaGetLastError(), "launching white balance");
    checkCudaError(cudaEventRecord(stageEvents[3]), "recording event 3");

    if (usePlanarLayout) {
        denoisePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                       imageWidth, imageHeight);
    } else {
        denoiseKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                 imageWidth, imageHeight);
    }
    checkCudaError(cudaGetLastError(), "launching denoise");
    checkCudaError(cudaEventRecord(stageEvents[4]), "recording event 4");

    if (usePlanarLayout) {
        gammaPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                                     imageWidth, imageHeight);
    } else {
        gammaKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                               imageWidth, imageHeight);
    }
    checkCudaError(cudaGetLastError(), "launching gamma");
    checkCudaError(cudaEventRecord(stageEvents[5]), "recording event 5");

    checkCudaError(cudaEventSynchronize(stageEvents[STAGE_COUNT]), "waiting for the pipeline");
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        checkCudaError(cudaEventElapsedTime(&stageMilliseconds[stage],
                                            stageEvents[stage], stageEvents[stage + 1]),
                       "reading stage time");
    }
}

// One warm-up run (thrown away), then TIMED_RUN_COUNT runs averaged per stage.
static void measureConfiguration(DeviceBuffers& buffers, const ImageMetadata& metadata,
                                 bool usePlanarLayout, int blockWidth, int blockHeight,
                                 cudaEvent_t stageEvents[], float averageMilliseconds[]) {
    dim3 blockShape(blockWidth, blockHeight);
    int blocksAcross = (metadata.imageWidth + blockWidth - 1) / blockWidth;
    int blocksDown = (metadata.imageHeight + blockHeight - 1) / blockHeight;
    dim3 gridShape(blocksAcross, blocksDown);

    float thisRunMilliseconds[STAGE_COUNT];
    runPipelineOnce(buffers, metadata, usePlanarLayout, gridShape, blockShape,
                    stageEvents, thisRunMilliseconds);

    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        averageMilliseconds[stage] = 0.0f;
    }
    for (int runNumber = 0; runNumber < TIMED_RUN_COUNT; runNumber++) {
        runPipelineOnce(buffers, metadata, usePlanarLayout, gridShape, blockShape,
                        stageEvents, thisRunMilliseconds);
        for (int stage = 0; stage < STAGE_COUNT; stage++) {
            averageMilliseconds[stage] = averageMilliseconds[stage] + thisRunMilliseconds[stage];
        }
    }
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        averageMilliseconds[stage] = averageMilliseconds[stage] / TIMED_RUN_COUNT;
    }
}

int main() {
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    DeviceBuffers buffers;
    allocateDeviceBuffers(buffers, pixelCount);
    checkCudaError(cudaMemcpy(buffers.deviceBayer, bayerPixels.data(), buffers.bayerBytes,
                              cudaMemcpyHostToDevice), "copying Bayer data to the GPU");

    cudaEvent_t stageEvents[STAGE_COUNT + 1];
    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventCreate(&stageEvents[eventNumber]), "creating stage event");
    }

    const char* configurationNames[CONFIGURATION_COUNT] = {
        "A interleaved  1x32",
        "B interleaved 16x16",
        "C interleaved  32x8",
        "D planar      16x16",
        "E planar       32x8"
    };
    bool configurationIsPlanar[CONFIGURATION_COUNT] = {false, false, false, true, true};
    int configurationBlockWidth[CONFIGURATION_COUNT] = {1, 16, 32, 16, 32};
    int configurationBlockHeight[CONFIGURATION_COUNT] = {32, 16, 8, 16, 8};

    // Table header. std::setw(n) pads the next printed item to n characters,
    // which lines the columns up.
    std::cout << std::fixed << std::setprecision(3);
    std::cout << "Average kernel times in ms (" << TIMED_RUN_COUNT << " warm runs each)" << std::endl;
    std::cout << std::setw(21) << "configuration"
              << std::setw(10) << "black" << std::setw(10) << "demosaic"
              << std::setw(10) << "white b." << std::setw(10) << "denoise"
              << std::setw(10) << "gamma" << std::setw(10) << "TOTAL" << std::endl;

    for (int configuration = 0; configuration < CONFIGURATION_COUNT; configuration++) {
        float averageMilliseconds[STAGE_COUNT];
        measureConfiguration(buffers, metadata,
                             configurationIsPlanar[configuration],
                             configurationBlockWidth[configuration],
                             configurationBlockHeight[configuration],
                             stageEvents, averageMilliseconds);

        float totalMilliseconds = 0.0f;
        std::cout << std::setw(21) << configurationNames[configuration];
        for (int stage = 0; stage < STAGE_COUNT; stage++) {
            std::cout << std::setw(10) << averageMilliseconds[stage];
            totalMilliseconds = totalMilliseconds + averageMilliseconds[stage];
        }
        std::cout << std::setw(10) << totalMilliseconds << std::endl;
    }

    // The last configuration run was E (planar 32x8), so the output buffer now
    // holds its result. Save it so we can check it against the CPU.
    std::vector<unsigned char> outputBytes(buffers.outputBytes);
    checkCudaError(cudaMemcpy(outputBytes.data(), buffers.deviceOutput, buffers.outputBytes,
                              cudaMemcpyDeviceToHost), "copying output to the CPU");

    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventDestroy(stageEvents[eventNumber]), "destroying stage event");
    }
    freeDeviceBuffers(buffers);

    bool writeOk = writeRgbFile("output/gpu_planar_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    std::cout << "Saved output/gpu_planar_output.bin (configuration E)" << std::endl;
    return 0;
}


## Step 3: Compile
This builds the experiment and also rebuilds `compare_outputs`, because Colab's `/content` disk is wiped between sessions.

**Expected:** no output.

In [ ]:
!nvcc -O2 -arch=sm_75 -o /content/gpu_coalescing gpu_coalescing_experiment_main.cu gpu_planar_kernels.cu gpu_buffers.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp
!g++ -O2 -std=c++17 -o /content/compare_outputs compare_outputs.cpp isp_io.cpp

## Step 4: Run the experiment (CHECKPOINT 1)
**Expected:** a table with 5 rows (A to E) and a time for each stage plus a TOTAL, then `Saved output/gpu_planar_output.bin`. It takes a few seconds.

**Paste the whole table to me.** I'll explain what it shows.

In [ ]:
!/content/gpu_coalescing

## Step 5: Check the planar version is still correct (CHECKPOINT 2)
**Expected:** `RESULT: PASS`.

In [ ]:
!/content/compare_outputs output/cpu_output.bin output/gpu_planar_output.bin